# Imports and installation

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install transformers
!pip install bz2

In [ ]:
!pip install wiki_dump_reader

In [ ]:
import transformers
import bz2
import re
import gc
import torch
import random
import os
import time
import datetime
import numpy as np
import pandas as pd 
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import XLMRobertaForSequenceClassification, XLMRobertaTokenizer, AdamW, BertConfig, get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, random_split, DataLoader, RandomSampler, SequentialSampler
import glob
from wiki_dump_reader import Cleaner, iterate
from nltk import sent_tokenize
from math import ceil
import csv
import pickle

# Functions for loading, preprocessing and tokenizing data

In [ ]:
# Load data, select n_texts random rows and split up labels and texts
def load_and_preprocess_data(data_path, n_texts, seed_val=42):
    if data_path.endswith(".bz2"):
        data_file = bz2.BZ2File(data_path)
        data_file_lines = data_file.readlines()
        del data_file # Clean off old variable to save memory
    else:
        with open(data_path, "r", encoding="utf8") as data_file:
            data_file_lines = data_file.readlines()
    print("Loaded {} lines".format(len(data_file_lines)))
    
    random.seed(seed_val)
    data_file_lines = random.sample(data_file_lines, n_texts)
    data_file_lines = [x.decode('utf-8') for x in data_file_lines]
    
    # Split labels and texts
    labels = [0 if x.split(' ')[0] == '__label__1' else 1 for x in data_file_lines]
    texts = [x.split(' ', 1)[1][:-1] for x in data_file_lines]
    
    # Clean up a bit
    del data_file_lines
    gc.collect()
    
    # Clean URL:s
    for i in range(len(texts)):
        if 'www.' in texts[i] or 'http:' in texts[i] or 'https:' in texts[i] or '.com' in texts[i]:
            texts[i] = re.sub(r"([^ ]+(?<=\.[a-z]{3}))", "<url>", texts[i])
    
    return texts, labels

In [ ]:
def load_neutral_wiki_data(data_path, nr_neutral_texts, max_sentence_per_text):
    cleaner = Cleaner()
    nr_neutral_resource = len(glob.glob1(data_path, "*.xml"))
    nr_each_resource = ceil(nr_neutral_texts / nr_neutral_resource)
    neutral_texts = []
    for filename in os.listdir(data_path):
        current_nr_texts = 0
        if filename.endswith(".xml"):
            for title, text in iterate(os.path.join(data_path, filename)):
                if current_nr_texts < nr_each_resource and len(neutral_texts) < nr_neutral_texts:
                    text = cleaner.clean_text(text)
                    cleaned_text, _ = cleaner.build_links(text)
                    sentences = sent_tokenize(cleaned_text)
                    if len(sentences) > 1:
                        sentences = [sentence for sentence in sentences if len(sentence.splitlines()) == 1]
                        if sentences:
                            nr_sentences = random.randint(1, max_sentence_per_text)
                            sentences = sentences[:nr_sentences]
                            text = sentences[0]
                            for sentence in sentences[1:]:
                                text += " " + sentence
                            neutral_texts.append(text)
                            current_nr_texts += 1
                else:
                    break
        else:
            continue 
    if len(neutral_texts) < nr_neutral_texts:
        print("There is not enough data for the specified number of neutral texts.")
    neutral_labels = [2] * len(neutral_texts)
    with open("./{}-texts_from_wiki_dump.pkl".format(nr_neutral_texts), 'wb') as f:
        pickle.dump(neutral_texts, f)
    return neutral_texts, neutral_labels

In [ ]:
# Tokenize all of the texts and map the tokens to their word IDs
def tokenize_data(tokenizer, texts):
    input_ids = []
    attention_masks = []
    
    # For every text...
    for text in tqdm(texts):
        # `encode_plus` will:
        #   (1) Tokenize the text.
        #   (2) Prepend the `[CLS]` token to the start.
        #   (3) Append the `[SEP]` token to the end.
        #   (4) Map tokens to their IDs.
        #   (5) Pad or truncate the text to `max_length`
        #   (6) Create attention masks for [PAD] tokens.
        encoded_dict = tokenizer.encode_plus(
                            text,                      # Text to encode.
                            add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                            max_length = 128,           # Pad & truncate all texts.
                            pad_to_max_length = True,
                            return_attention_mask = True,   # Construct attn. masks.
                            return_tensors = 'pt',     # Return pytorch tensors.
                            truncation = True          # Truncate too long texts
                       )

        # Add the encoded text to the list.    
        input_ids.append(encoded_dict['input_ids'])

        # And its attention mask (simply differentiates padding from non-padding).
        attention_masks.append(encoded_dict['attention_mask'])

    # Convert the lists into tensors.
    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    
    return input_ids, attention_masks

In [ ]:
# Functions to save and load the preprocessed and tokenized data
def save_tokenized_data(data_pp_dir, input_ids, attention_masks, labels, texts):
    torch.save(input_ids, os.path.join(data_pp_dir, "input_ids.pt"))
    torch.save(attention_masks, os.path.join(data_pp_dir, "attention_masks.pt"))
    torch.save(labels, os.path.join(data_pp_dir, "labels.pt"))
    with open(os.path.join(data_pp_dir, "texts.txt"), "w", encoding="utf8") as texts_file:
        for text in texts:
            print(text, file=texts_file)
    print("Saved tokenized data to: {}".format(data_pp_dir))
            
def load_tokenized_data(data_pp_dir):
    input_ids = torch.load(os.path.join(data_pp_dir, "input_ids.pt"))
    attention_masks = torch.load(os.path.join(data_pp_dir, "attention_masks.pt"))
    labels = torch.load(os.path.join(data_pp_dir, "labels.pt"))
    with open(os.path.join(data_pp_dir, "texts.txt"), "r") as texts_file:
        # all_lines = texts_file.read_lines()
        # texts = [line.strip() for line in all_lines]
        texts = [line.rstrip() for line in texts_file]
    return input_ids, attention_masks, labels, texts

# Functions for predicting

In [ ]:
# Function to load an already fine-tuned model
def load_model(model_dir):
    print("Loading tokenizer and model from: {}".format(model_dir))
    tokenizer = XLMRobertaTokenizer.from_pretrained(model_dir)
    model_loaded = XLMRobertaForSequenceClassification.from_pretrained(model_dir)
    return tokenizer, model_loaded

In [ ]:
def predict_in_batches(input_ids, attention_masks, labels):
    # Create the DataLoader.
    prediction_data = TensorDataset(input_ids, attention_masks, labels)
    prediction_sampler = SequentialSampler(prediction_data)
    prediction_dataloader = DataLoader(prediction_data, sampler=prediction_sampler, batch_size=batch_size)
    
    # Prediction on test set
    print('Predicting labels for {:,} test texts...'.format(len(input_ids)))

    # Put model in evaluation mode
    model.eval()

    # Tracking variables 
    predictions , true_labels = [], []

    # Predict 
    for batch in prediction_dataloader:
      # Add batch to GPU
      batch = tuple(t.to(device) for t in batch)

      # Unpack the inputs from our dataloader
      b_input_ids, b_input_mask, b_labels = batch

      # Telling the model not to compute or store gradients, saving memory and 
      # speeding up prediction
      with torch.no_grad():
          # Forward pass, calculate logit predictions
          outputs = model(b_input_ids, token_type_ids=None, 
                          attention_mask=b_input_mask)

      logits = outputs[0]

      # Move logits and labels to CPU
      logits = logits.detach().cpu().numpy()
      label_ids = b_labels.to('cpu').numpy()

      # Store predictions and true labels
      predictions.append(logits)
      true_labels.append(label_ids)

    return predictions, true_labels

In [ ]:
def predict_sentiment(model, device, texts):
    # Put model in evaluation mode
    model.eval()
    
    # Tokenize all of the texts and map the tokens to their word IDs.
    input_ids, attention_masks = tokenize_data(tokenizer, texts)
    
    # Move to correct device
    model = model.to(device)
    input_ids = input_ids.to(device)
    attention_masks = attention_masks.to(device)
    
    # Predict sentiment
    with torch.no_grad():
        # Forward pass, calculate logit predictions
        outputs = model(input_ids, token_type_ids=None, attention_mask=attention_masks)
        logits = outputs[0]
        logits = logits.detach().cpu().numpy()
        predicted_labels = np.argmax(logits, axis=1)

    return predicted_labels

# Functions for training

In [ ]:
# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)


# Function to format elapsed time in printouts
def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))


def print_params(model):
    # Get all of the model's parameters as a list of tuples.
    params = list(model.named_parameters())

    print('The XLMRoberta model has {:} different named parameters.\n'.format(len(params)))

    print('==== Embedding Layer ====\n')

    for p in params[0:5]:
        print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

    print('\n==== First Transformer ====\n')

    for p in params[5:21]:
        print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

    print('\n==== Output Layer ====\n')

    for p in params[-4:]:
        print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

In [ ]:
def training(model, epochs, train_dataloader, validation_dataloader, optimizer, scheduler, seed_val=42):
    # This training code is based on the `run_glue.py` script here:
    # https://github.com/huggingface/transformers/blob/5bfcd0485ece086ebcbed2d008813037968a9e58/examples/run_glue.py#L128

    # Set the seed value all over the place to make this reproducible.
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

    model.to(device)

    # We'll store a number of quantities such as training and validation loss, 
    # validation accuracy, and timings.
    training_stats = []

    # Measure the total training time for the whole run.
    total_t0 = time.time()

    # For each epoch...
    for epoch_i in range(0, epochs):

        # ========================================
        #               Training
        # ========================================

        # Perform one full pass over the training set.

        print("")
        print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
        print('Training...')

        # Measure how long the training epoch takes.
        t0 = time.time()

        # Reset the total loss for this epoch.
        total_train_loss = 0

        # Put the model into training mode. Don't be mislead--the call to 
        # `train` just changes the *mode*, it doesn't *perform* the training.
        # `dropout` and `batchnorm` layers behave differently during training
        # vs. test (source: https://stackoverflow.com/questions/51433378/what-does-model-train-do-in-pytorch)
        model.train()

        # For each batch of training data...
        for step, batch in enumerate(train_dataloader):

            # Progress update every 40 batches.
            if step % 40 == 0 and not step == 0:
                # Calculate elapsed time in minutes.
                elapsed = format_time(time.time() - t0)

                # Report progress.
                print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

            # Unpack this training batch from our dataloader. 
            #
            # As we unpack the batch, we'll also copy each tensor to the GPU using the 
            # `to` method.
            #
            # `batch` contains three pytorch tensors:
            #   [0]: input ids 
            #   [1]: attention masks
            #   [2]: labels 
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            # Always clear any previously calculated gradients before performing a
            # backward pass. PyTorch doesn't do this automatically because 
            # accumulating the gradients is "convenient while training RNNs". 
            # (source: https://stackoverflow.com/questions/48001598/why-do-we-need-to-call-zero-grad-in-pytorch)
            model.zero_grad()        

            # Perform a forward pass (evaluate the model on this training batch).
            # The documentation for this `model` function is here: 
            # https://huggingface.co/transformers/v2.2.0/model_doc/bert.html#transformers.BertForSequenceClassification
            # It returns different numbers of parameters depending on what arguments
            # arge given and what flags are set. For our useage here, it returns
            # the loss (because we provided labels) and the "logits"--the model
            # outputs prior to activation.
            l = model(b_input_ids, 
                                 token_type_ids=None, 
                                 attention_mask=b_input_mask, 
                                 labels=b_labels)
            loss = l[0]
            logits = l[1]
            # Accumulate the training loss over all of the batches so that we can
            # calculate the average loss at the end. `loss` is a Tensor containing a
            # single value; the `.item()` function just returns the Python value 
            # from the tensor.
            total_train_loss += loss.item()

            # Perform a backward pass to calculate the gradients.
            loss.backward()

            # Clip the norm of the gradients to 1.0.
            # This is to help prevent the "exploding gradients" problem.
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Update parameters and take a step using the computed gradient.
            # The optimizer dictates the "update rule"--how the parameters are
            # modified based on their gradients, the learning rate, etc.
            optimizer.step()

            # Update the learning rate.
            scheduler.step()

        # Calculate the average loss over all of the batches.
        avg_train_loss = total_train_loss / len(train_dataloader)            

        # Measure how long this epoch took.
        training_time = format_time(time.time() - t0)

        print("")
        print("  Average training loss: {0:.2f}".format(avg_train_loss))
        print("  Training epcoh took: {:}".format(training_time))

        # ========================================
        #               Validation
        # ========================================
        # After the completion of each training epoch, measure our performance on
        # our validation set.

        print("")
        print("Running Validation...")

        t0 = time.time()

        # Put the model in evaluation mode--the dropout layers behave differently
        # during evaluation.
        model.eval()

        # Tracking variables 
        total_eval_accuracy = 0
        total_eval_loss = 0
        nb_eval_steps = 0

        # Evaluate data for one epoch
        for batch in validation_dataloader:

            # Unpack this training batch from our dataloader. 
            #
            # As we unpack the batch, we'll also copy each tensor to the GPU using 
            # the `to` method.
            #
            # `batch` contains three pytorch tensors:
            #   [0]: input ids 
            #   [1]: attention masks
            #   [2]: labels 
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            # Tell pytorch not to bother with constructing the compute graph during
            # the forward pass, since this is only needed for backprop (training).
            with torch.no_grad():        

                # Forward pass, calculate logit predictions.
                # token_type_ids is the same as the "segment ids", which 
                # differentiates text 1 and 2 in 2-text tasks.
                # The documentation for this `model` function is here: 
                # https://huggingface.co/transformers/v2.2.0/model_doc/bert.html#transformers.BertForSequenceClassification
                # Get the "logits" output by the model. The "logits" are the output
                # values prior to applying an activation function like the softmax.
                l = model(b_input_ids, 
                                       token_type_ids=None, 
                                       attention_mask=b_input_mask,
                                       labels=b_labels)
                loss = l[0]
                logits = l[1]

            # Accumulate the validation loss.
            total_eval_loss += loss.item()

            # Move logits and labels to CPU
            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.to('cpu').numpy()

            # Calculate the accuracy for this batch of test texts, and
            # accumulate it over all batches.
            total_eval_accuracy += flat_accuracy(logits, label_ids)


        # Report the final accuracy for this validation run.
        avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
        print("  Accuracy: {0:.2f}".format(avg_val_accuracy))

        # Calculate the average loss over all of the batches.
        avg_val_loss = total_eval_loss / len(validation_dataloader)

        # Measure how long the validation run took.
        validation_time = format_time(time.time() - t0)

        print("  Validation Loss: {0:.2f}".format(avg_val_loss))
        print("  Validation took: {:}".format(validation_time))

        # Record all statistics from this epoch.
        training_stats.append(
            {
                'epoch': epoch_i + 1,
                'Training Loss': avg_train_loss,
                'Valid. Loss': avg_val_loss,
                'Valid. Accur.': avg_val_accuracy,
                'Training Time': training_time,
                'Validation Time': validation_time
            }
        )
        output_dir = 'xlm-roberta_model_save_e{}'.format(epoch_i)

        # Create output directory if needed
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        print("Saving model to %s" % output_dir)

        # Save a trained model, configuration and tokenizer using `save_pretrained()`.
        # They can then be reloaded using `from_pretrained()`
        model_to_save = model.module if hasattr(model, 'module') else model  # Take care of distributed/parallel training
        model_to_save.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)

    print("")
    print("Training complete!")

    print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))
    
    return model, training_stats

In [ ]:
def visualize_training_stats(training_stats):
    # Display floats with two decimal places.
    pd.set_option('precision', 2)

    # Create a DataFrame from our training statistics.
    df_stats = pd.DataFrame(data=training_stats)

    # Use the 'epoch' as the row index.
    df_stats = df_stats.set_index('epoch')

    # A hack to force the column headers to wrap.
    #df = df.style.set_table_styles([dict(selector="th",props=[('max-width', '70px')])])

    # Display the table.
    print(df_stats)
    
    # Use plot styling from seaborn.
    sns.set(style='darkgrid')

    # Increase the plot size and font size.
    sns.set(font_scale=1.5)
    plt.rcParams["figure.figsize"] = (12,6)

    # Plot the learning curve.
    plt.plot(df_stats['Training Loss'], 'b-o', label="Training")
    plt.plot(df_stats['Valid. Loss'], 'g-o', label="Validation")

    # Label the plot.
    plt.title("Training & Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.xticks([x + 1 for x in df_stats.index])

    plt.show()

# Let's start

## Check device to use

In [ ]:
if torch.cuda.is_available():    

    # Tell PyTorch to use the GPU.    
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

## Load train data

In [ ]:
# os.rmdir("preprocessed_train_data_1000")

In [ ]:
data_dir = "../input/amazonreviews"
neutral_texts_dir = "../input/wiki-200k-dump"
train_file_name = "train.ft.txt.bz2"
n_train = 450000
n_neutral = 50000
train_data_pp_dir = "preprocessed_train_data_n{}".format(n_train)

if os.path.exists(train_data_pp_dir):
    print("Loading already preprocessed and tokenized data from: {}".format(train_data_pp_dir))
    train_input_ids, train_attention_masks, train_labels, train_texts = load_tokenized_data(train_data_pp_dir)
else:
    if os.path.exists(neutral_texts_dir):
        print("Reading neutral texts from pickle file...")
        with open("../input/wiki-200k-dump/200000-texts_from_wiki_dump.pkl", 'rb') as f:
            neutral_texts = pickle.load(f)
            neutral_texts = neutral_texts[:n_neutral]
            if len(neutral_texts) < n_neutral:
                print("There is not enough data for the specified number of neutral texts.")
                act_n_neutral = len(neutral_texts)
            else:
                act_n_neutral = n_neutral
            neutral_labels = [2] * act_n_neutral
    else:
        print("Loading neutral texts from wiki dump...")
        neutral_start_time = time.time()
        neutral_texts, neutral_labels = load_neutral_wiki_data("../input/neutral-resource", n_neutral, 3)
        print("Loading neutral texts took {}".format(time.time() - neutral_start_time))
        
    original_train_data_dir = os.path.join(data_dir, train_file_name)
    print("Loading and preprocessing data from: {}".format(original_train_data_dir))
    sentiment_start_time = time.time()
    train_texts, train_labels = load_and_preprocess_data(original_train_data_dir, n_train)
    print("Loading sentiment texts took {}".format(time.time() - sentiment_start_time))
    train_texts = train_texts + neutral_texts
    train_labels = train_labels + neutral_labels
    
    print("Tokenizing data")
    tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base', do_lower_case=True)
    train_input_ids, train_attention_masks = tokenize_data(tokenizer, train_texts)
    train_labels = torch.tensor(train_labels)
    
    os.makedirs(train_data_pp_dir)
    print("Saving tokenized data to: {}".format(train_data_pp_dir))
    save_tokenized_data(train_data_pp_dir, train_input_ids, train_attention_masks, train_labels, train_texts)
print(len(train_texts))
print(len(train_labels))

print("Texts: {}".format(len(train_texts)))
print("Unique labels: {}".format(len(set(train_labels))))
for i in range(3):
    print(train_labels[i], train_texts[i], "\n")
    
# Print text 0, now as a list of IDs.
print('Original text: ', train_texts[0])
print('Token IDs:', train_input_ids[0])
print('Label:', train_labels[0])
print(train_input_ids.shape)

# Training

## Split data in train and validation set

In [ ]:
# Combine the training inputs into a TensorDataset.
dataset = TensorDataset(train_input_ids, train_attention_masks, train_labels)

# Create a 90-10 train-validation split.

# Calculate the number of samples to include in each set.
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

# Divide the dataset by randomly selecting samples.
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print('{:>5,} training samples'.format(train_size))
print('{:>5,} validation samples'.format(val_size))

## Create everything needed for the training

In [ ]:
# The DataLoader needs to know our batch size for training, so we specify it 
# here. For fine-tuning BERT on a specific task, the authors recommend a batch 
# size of 16 or 32.
batch_size = 32


# Number of training epochs. The BERT authors recommend between 2 and 4. 
# We chose to run for 4, but we'll see later that this may be over-fitting the
# training data.
epochs = 2


# Create the DataLoaders for our training and validation sets.
# We'll take training samples in random order. 
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

# Total number of training steps is [number of batches] x [number of epochs]. 
# (Note that this is not the same as the number of training samples).
total_steps = len(train_dataloader) * epochs

# For validation the order doesn't matter, so we'll just read them sequentially.
validation_dataloader = DataLoader(
            val_dataset, # The validation samples.
            sampler = SequentialSampler(val_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

# Load BertForSequenceClassification, the pretrained BERT model with a single 
# linear classification layer on top. 
model = XLMRobertaForSequenceClassification.from_pretrained(
    "xlm-roberta-base", # Use the 12-layer BERT model, with an uncased vocab.
    num_labels = 3, # The number of output labels--2 for binary classification.
                    # You can increase this for multi-class tasks.   
    output_attentions = False, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)

print_params(model)

# Note: AdamW is a class from the huggingface library (as opposed to pytorch) 
# The 'W' might stand for 'Weight Decay fix"
optimizer = AdamW(model.parameters(),
                  lr = 2e-5, # args.learning_rate - default is 5e-5, our notebook had 2e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8.
                )

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

## Let's train

In [ ]:
model, training_stats = training(model, epochs, train_dataloader, validation_dataloader, optimizer, scheduler, seed_val=42)
visualize_training_stats(training_stats)

# Evaluation

## Evaluate on test set

In [ ]:
test_file_name = "test.ft.txt.bz2"
n_test = 1000

print("Loading, preprocessing and shuffling test data")
test_texts, test_labels = load_and_preprocess_data(os.path.join(data_dir, test_file_name), n_test)
test_zip = list(zip(test_texts, test_labels))
random.shuffle(test_zip)
test_texts, test_labels = zip(*test_zip)

print("Tokenizing")
test_input_ids, test_attention_masks = tokenize_data(tokenizer, test_texts)
test_labels = torch.tensor(test_labels)


print("Evaluating model on test data")
predictions, true_labels = predict_in_batches(test_input_ids, test_attention_masks, test_labels)
print("Test accuracy: {}".format(flat_accuracy(predictions[0], true_labels[0])))

## Predict sentiment for some new texts

In [ ]:
some_sentences = ["jag tyckte väldigt mycket om maten, den var god", "The hotel was not really that nice", "They had a large tv room,", "The capital of France is Paris.",
                  "Je ne l'ai pas du tout aimé, le personnel étais impoli et la nourriture froide. ", "der beste Film aller Zeiten", "It's his best movie ever." ,"I went to this hotel last year"]
some_predictions = predict_sentiment(model, device, some_sentences)
print(some_predictions)
for i in range(len(some_sentences)):
    if some_predictions[i] == 1:
        prediction = "positive"
    elif some_predictions[i] == 0:
        prediction = "negative"
    else:
        prediction = "neutral"
    print(prediction, " ", some_sentences[i], "\n")